In [0]:
%sql
-- Sample of 20 rows to see data structure

SELECT * FROM workspace.aml_bronze.raw_transactions LIMIT 20


In [0]:
%sql
-- Total transactions and laundering count

SELECT COUNT(*) AS total_txns, SUM(is_laundering) AS laundering_txns 
FROM workspace.aml_bronze.raw_transactions

In [0]:
%sql
-- Top 10 largest transactions

SELECT * FROM workspace.aml_bronze.raw_transactions 
ORDER BY amount_paid DESC 
LIMIT 10

In [0]:
%sql
-- First and last transaction dates

SELECT 
  MIN(TO_TIMESTAMP(timestamp, 'yyyy/MM/dd HH:mm')) AS first_date,
  MAX(TO_TIMESTAMP(timestamp, 'yyyy/MM/dd HH:mm')) AS last_date
FROM workspace.aml_bronze.raw_transactions

In [0]:
%sql
-- Laundering rate by payment method

SELECT payment_format, COUNT(*) AS txn_count, AVG(is_laundering) AS laundering_rate 
FROM workspace.aml_bronze.raw_transactions 
GROUP BY payment_format

In [0]:
%sql
-- Top 10 accounts that sent the most money

SELECT from_account, SUM(amount_paid) AS total_sent, COUNT(*) AS txn_count 
FROM workspace.aml_bronze.raw_transactions 
GROUP BY from_account 
ORDER BY total_sent DESC 
LIMIT 10

In [0]:
%sql
-- High-value accounts (average > 1M, 5+ transactions)

SELECT from_account, AVG(amount_paid) AS avg_paid, COUNT(*) AS txn_count
FROM workspace.aml_bronze.raw_transactions 
GROUP BY from_account 
HAVING AVG(amount_paid) > 1000000 AND COUNT(*) >= 5

In [0]:
%sql
-- Currency mismatch and laundering rates analysis

SELECT 
  AVG(CASE WHEN receiving_currency <> payment_currency THEN 1 ELSE 0 END) AS currency_mismatch_rate,
  AVG(CASE WHEN receiving_currency <> payment_currency THEN is_laundering END) AS launder_rate_mismatch,
  AVG(CASE WHEN receiving_currency = payment_currency THEN is_laundering END) AS launder_rate_match
FROM workspace.aml_bronze.raw_transactions

In [0]:
%sql
-- Using CTE to add currency mismatch flag, then aggregate

WITH flagged AS (
  SELECT *, 
    CASE WHEN receiving_currency <> payment_currency THEN 1 ELSE 0 END AS is_currency_mismatch
  FROM workspace.aml_bronze.raw_transactions
)
SELECT 
  is_currency_mismatch,
  COUNT(*) AS txn_count,
  AVG(is_laundering) AS laundering_rate
FROM flagged
GROUP BY is_currency_mismatch

In [0]:
%sql
-- CTE with timestamp conversion and basic features

WITH clean AS (
  SELECT *,
    TO_TIMESTAMP(timestamp, 'yyyy/MM/dd HH:mm') AS transaction_timestamp,
    CASE WHEN from_account = to_account THEN 1 ELSE 0 END AS is_same_account,
    CASE WHEN receiving_currency <> payment_currency THEN 1 ELSE 0 END AS is_currency_mismatch,
    ABS(amount_received - amount_paid) AS amount_discrepancy
  FROM workspace.aml_bronze.raw_transactions
)
SELECT * FROM clean LIMIT 20

In [0]:
%sql
-- Running count of transactions per from_account (last 100 rows)

SELECT 
  from_account,
  timestamp,
  amount_paid,
  COUNT(*) OVER (
    PARTITION BY from_account
    ORDER BY timestamp
    ROWS BETWEEN 10 PRECEDING AND CURRENT ROW
  ) AS txn_count_last_10
FROM workspace.aml_bronze.raw_transactions
LIMIT 100

In [0]:
%sql
-- Rolling sum of amount_paid per from_account (last 100 rows)

SELECT 
  from_account,
  timestamp,
  amount_paid,
  SUM(amount_paid) OVER (
    PARTITION BY from_account
    ORDER BY timestamp
    ROWS BETWEEN 10 PRECEDING AND CURRENT ROW
  ) AS amount_sum_last_10
FROM workspace.aml_bronze.raw_transactions
LIMIT 100

In [0]:
%sql
-- Count of unique senders per to_account (last 100 rows)

SELECT 
  to_account,
  from_account,
  timestamp,
  SIZE(COLLECT_SET(from_account) OVER (
    PARTITION BY to_account
    ORDER BY timestamp
    ROWS BETWEEN 10 PRECEDING AND CURRENT ROW
  )) AS unique_senders_last_10
FROM workspace.aml_bronze.raw_transactions
LIMIT 100

In [0]:
%sql
-- Count of unique recipients per from_account (last 100 rows)

SELECT 
  from_account,
  to_account,
  timestamp,
  SIZE(COLLECT_SET(to_account) OVER (
    PARTITION BY from_account
    ORDER BY timestamp
    ROWS BETWEEN 10 PRECEDING AND CURRENT ROW
  )) AS unique_recipients_last_10
FROM workspace.aml_bronze.raw_transactions
LIMIT 100